# Mauna Loa CO2 data exploration 

In this notebook, we explore the use of a moving average to remove seasonal variations and focus on the long-term variations in the CO2 concentration measured at Mauna Loa. 

Data source: https://scrippsco2.ucsd.edu/data/atmospheric_co2/primary_mlo_co2_record.html

In [ ]:
import numpy as np
import pandas as pd
from scipy import signal
import matplotlib.pyplot as plt
import ipywidgets

In [ ]:
from matplotlib import rcParams
rcParams["font.size"]=14

## What are our data? 

The data file below contains 10 columns.  Columns 1-4 give the dates in several redundant  formats. Column 5 below gives monthly Mauna Loa CO2 concentrations in micro-mol CO2 per  mole (ppm), reported on the 2008A SIO manometric mole fraction scale.  This is the  standard version of the data most often sought.  The monthly values have been adjusted  to 24:00 hours on the 15th of each month.  Column 6 gives the same data after a seasonal adjustment to remove the quasi-regular seasonal cycle.  The adjustment involves  subtracting from the data a 4-harmonic fit with a linear gain factor.  Column 7 is a  smoothed version of the data generated from a stiff cubic spline function plus 4-harmonic  functions with linear gain.  Column 8 is the same smoothed version with the seasonal  cycle removed.  Column 9 is identical to Column 5 except that the missing values from  Column 5 have been filled with values from Column 7.  Column 10 is identical to Column 6   except missing values have been filled with values from Column 8.  Missing values are  denoted by -99.99                                                                         

In [ ]:
co2_data_source = "https://scrippsco2.ucsd.edu/assets/data/atmospheric/stations/in_situ_co2/monthly/monthly_in_situ_co2_mlo.csv"

In [ ]:
# read data and do some data cleaning 
co2_data = pd.read_csv(
    co2_data_source, skiprows=np.arange(0, 64), na_values="-99.99",
    header=None
)

In [ ]:
co2_data.columns = [
    "year", "month", "date (excel)", "date", "co2", "seasonally adjusted",
    "fit", "seasonally adjusted fit", "co2 filled", "seasonally adjusted filled", 
    "station"
]
co2_data

In [ ]:
def plot_co2_data(data=co2_data, ax=None, xlim=None, ylim=None):
    if ax is None: 
        fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    ax.plot(
        data["date"], data["co2"], 
        label="CO$_2$ [ppm]"
    )
    ax.plot(
        data["date"], data["seasonally adjusted"], 
        label="seasonally adjusted",
    )
    ax.set_xlabel("Year")
    ax.set_ylabel("CO$_2$ Concentration (ppm)")
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.grid()
    return ax

In [ ]:
ax = plot_co2_data()
ax.legend()

## Moving average to remove seasonal variations

Does using a moving average produce similar results as the seasonally adjusted data provided by scripps? 

In [ ]:
def moving_average(data, window_size=1):
    n_data = len(data)
    average = np.full(n_data, np.nan)
    if window_size == 1: 
        return data
    half_window = window_size // 2
    for i in range(half_window, n_data - half_window):
        average[i] = np.mean(data[i - half_window: i + half_window])
    return average

In [ ]:
def plot_co2_average(window_size=1, xmin=None, xmax=None, ymin=None, ymax=None):
    averaged_co2 = moving_average(co2_data["co2"], window_size)

    ax=plot_co2_data(co2_data)
    ax.plot(
        co2_data["date"], averaged_co2, 
        label=f"moving average: {window_size}"
    )
    ax.legend()
    ax.set_xlim([xmin, xmax])
    ax.set_ylim([ymin, ymax])

In [ ]:
ipywidgets.interactive(
    plot_co2_average, 
    window_size=ipywidgets.IntSlider(min=1, max=24, value=1),
    xmin=ipywidgets.FloatText(value=1955), 
    xmax=ipywidgets.FloatText(value=2026),
    ymin=ipywidgets.FloatText(value=310),
    ymax=ipywidgets.FloatText(value=450)
)